# SafeCrop · Photo check: what is wrong with this photo, and what should the app do? (Google Colab)

Before diagnosing, the app looks at the photo itself. This notebook:

1. **Measures 16 qualities** of a photo: brightness, very dark areas, contrast, colour, grain (overall and coloured speckles), sharpness, fine detail, edges, smearing in one direction, JPEG blockiness and how much of the photo is leaf.
2. **Compares them with our clean photos.** Anything outside the range of 98% of our clean photos is flagged in plain words, such as *too dark*, *grainy*, *blurry* or *blocky*. A photo can have several problems at once.
3. **Names the main problem and how bad it is.** A random forest trained on our 840 images says which of the five problems (or none) the photo has, and the level (L0 to L3). On photos it never saw, it is right about **98%** of the time for the problem and **92%** for the level; its only mistakes are telling hand-shake blur from out-of-focus blur.
4. **Chooses the fix:** only JPEG compression gets a fix (resample), because it is the only fix our L3 variants test proved helpful for CLIP. Every other problem is named and the photo is left unchanged. When the forest is unsure (less than 60%), the list of problems decides instead: a photo flagged as blocky gets resample. The photo is shown before and after.

**One honest limit:** the forest learned from our photos, where each noisy photo has exactly one problem made by a computer. A real field photo can have several problems at once, or a kind of damage the forest has never seen. That's why the plain-word list (step 2) is shown next to the forest's answer, together with how sure the forest is.

### Before you run

1. A GPU isn't needed.
2. `noisy_images.zip` in your Drive (to learn from), as before.
3. **Photos to check:** set `CHECK_FOLDER` to a folder in your Drive, or leave `ASK_FOR_UPLOAD = True` to get an upload button. With `SHOW_EXAMPLES = True` it also checks one L3 photo of each noise type from our dataset.
4. **Runtime → Run all.** Reports and before/after pictures go to `SafeCrop/results/photo_check`.

In [ ]:
# Install anything missing (Colab already has most of it). Safe to run again.
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

def ensure(pkg, min_major=None):
    try:
        v = version(pkg)
        if min_major is not None and int(v.split(".")[0]) < min_major:
            raise PackageNotFoundError(pkg)
    except PackageNotFoundError:
        print("installing", pkg, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], check=True)

ensure("openpyxl")
ensure("scikit-learn")
ensure("scikit-image")
try:
    import cv2
except ImportError:
    ensure("opencv-python-headless")
print("scikit-learn", version("scikit-learn"))


In [ ]:
import os, re, io, csv, json, time, glob, math, random, shutil, zipfile, datetime, subprocess
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image as ShowImage
from tqdm.auto import tqdm

def short(e, n=300):
    s = str(e).replace("\n", " ")
    return s if len(s) <= n else s[:n] + " ..."

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("pandas", pd.__version__, "| numpy", np.__version__)

## Settings

You can leave these as they are.

In [ ]:
# ---- the images the check learns from ----
SOURCE = "auto"      # "auto": look in Google Drive, ask for an upload if not found | "drive" | "upload"
DATASET_PATH = ""
OUT_SUBDIR = "photo_check"

# ---- photos to check ----
CHECK_FOLDER = ""            # a folder in My Drive with photos to check, e.g. "SafeCrop/field_photos"
ASK_FOR_UPLOAD = True        # show an upload button for your own photos
SHOW_EXAMPLES = True         # also check one L3 photo of each noise type from our dataset
SEED = 299

## 1. Get the images to learn from

In [ ]:
WORK = "/content/safecrop_inference"
DATA_DIR = os.path.join(WORK, "data")
os.makedirs(DATA_DIR, exist_ok=True)

def mount_drive():
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Couldn't connect to Google Drive:", short(e, 200))
    return os.path.isdir("/content/drive/MyDrive")

def walk(top, max_depth):
    # os.walk that stops max_depth folders below `top` and skips hidden folders
    top = top.rstrip(os.sep)
    base = top.count(os.sep)
    for root, dirs, files in os.walk(top):
        depth = root.count(os.sep) - base
        dirs[:] = sorted(d for d in dirs if not d.startswith(".")) if depth < max_depth else []
        yield root, files

def is_dataset_folder(root, files):
    return "metadata.csv" in files and os.path.isdir(os.path.join(root, "L0"))

def find_dataset_folder(top):
    for root, files in walk(top, max_depth=6):
        if is_dataset_folder(root, files):
            return root
    return None

def unpack(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist()
                   if not m.startswith("__MACOSX") and not os.path.basename(m.rstrip("/")).startswith("._")]
        z.extractall(dest, members)

def prepare(path):
    # returns the folder that holds metadata.csv and L0..L3, unzipping first if needed
    if os.path.isdir(path):
        return find_dataset_folder(path)
    if not (os.path.isfile(path) and zipfile.is_zipfile(path)):
        return None
    dest = os.path.join(DATA_DIR, re.sub(r"[^\w.-]+", "_", os.path.splitext(os.path.basename(path))[0]))
    if os.path.exists(dest):
        shutil.rmtree(dest)
    print("Unzipping", os.path.basename(path), "...")
    unpack(path, dest)
    for _ in range(3):   # handles a zip inside the zip, e.g. the whole 'safecrop dataset' folder downloaded from Drive
        found = find_dataset_folder(dest)
        if found:
            return found
        inner = [os.path.join(r, f) for r, fs in walk(dest, 6) for f in fs if f.lower().endswith(".zip")]
        inner = [p for p in inner if "noisy" in os.path.basename(p).lower()] or inner
        if not inner:
            break
        for p in inner:
            unpack(p, os.path.splitext(p)[0])
            os.remove(p)
    return None

def find_in_drive():
    found = []
    for root, files in walk("/content/drive/MyDrive", max_depth=4):
        if is_dataset_folder(root, files):
            found.append(root)
        for f in files:
            fl = f.lower()
            if fl.endswith(".zip") and "noisy" in fl and "result" not in fl:
                found.append(os.path.join(root, f))
    return sorted(found, key=os.path.getmtime, reverse=True)

DRIVE_OK, dataset_dir, uploaded = False, None, []

if DATASET_PATH:
    if DATASET_PATH.startswith("/content/drive"):
        DRIVE_OK = mount_drive()
    dataset_dir = prepare(DATASET_PATH)
    if dataset_dir is None:
        print("No images found at DATASET_PATH:", DATASET_PATH)

if dataset_dir is None and SOURCE in ("auto", "drive"):
    DRIVE_OK = mount_drive()
    if DRIVE_OK:
        print("Searching your Google Drive for noisy_images ...")
        candidates = find_in_drive()
        for c in candidates:
            print("  found:", c)
        for c in candidates:
            dataset_dir = prepare(c)
            if dataset_dir:
                print("Using:", c)
                break
        if dataset_dir is None:
            print("No noisy_images.zip found in your Drive. (Files in 'Shared with me' only count if you add a shortcut to My Drive.)")

if dataset_dir is None and SOURCE in ("auto", "upload"):
    from google.colab import files
    print("Click 'Choose files' and pick noisy_images.zip. Without Drive, also add the answer files from "
          "earlier notebooks (gemini_3.5_flash_lite_predictions.csv, ..._help_both_predictions.csv).")
    os.makedirs(os.path.join(WORK, "uploads"), exist_ok=True)
    for name, data in files.upload().items():
        p = os.path.join(WORK, "uploads", name)
        with open(p, "wb") as f:
            f.write(data)
        uploaded.append(p)
    for p in uploaded:
        if p.lower().endswith(".zip"):
            dataset_dir = prepare(p)
            if dataset_dir:
                break

if dataset_dir is None:
    raise RuntimeError("No images found. Put noisy_images.zip in your Google Drive (or set SOURCE = 'upload') and run again.")

# ---- read the list of images ----
meta = pd.read_csv(os.path.join(dataset_dir, "metadata.csv"))
need = {"image", "photo_id", "disease", "level"}
if need - set(meta.columns):
    raise RuntimeError(f"metadata.csv is missing the columns {sorted(need - set(meta.columns))}")
for col in ["disease_name", "group", "origin", "noise"]:
    if col not in meta.columns:
        meta[col] = ""
meta = meta.fillna("")
meta["disease_name"] = [n if n else d for n, d in zip(meta.disease_name, meta.disease)]
meta["level"] = meta.level.astype(str)
meta["path"] = [os.path.join(dataset_dir, p) for p in meta.image]
missing = ~meta.path.map(os.path.exists)
if missing.any():
    print(f"Warning: {missing.sum()} images listed in metadata.csv are missing and will be skipped.")
    meta = meta[~missing]
meta = meta.drop_duplicates("image").sort_values(["disease", "photo_id", "level"]).reset_index(drop=True)

# ---- where results go ----
RESULTS_DIR = "/content/drive/MyDrive/SafeCrop/results" if DRIVE_OK else os.path.join(WORK, "results")
OUT_DIR = os.path.join(RESULTS_DIR, OUT_SUBDIR)       # everything this notebook writes
os.makedirs(OUT_DIR, exist_ok=True)
GEMINI_CACHE = os.path.join(RESULTS_DIR, "gemini_3.5_flash_lite_predictions.csv")   # Step 2: plain Gemini answers
HELP_CACHE = os.path.join(RESULTS_DIR, "gemini_help", "gemini_3.5_flash_lite_help_both_predictions.csv")   # Step 2b

def merge_upload(src, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    old = pd.read_csv(src, dtype=str, keep_default_na=False)
    if os.path.exists(dest):
        old = pd.concat([pd.read_csv(dest, dtype=str, keep_default_na=False), old])
    old.drop_duplicates(["image", "model"], keep="last").to_csv(dest, index=False)
    print("Using", os.path.basename(src))

for p in uploaded:   # answers from earlier notebooks, when working without Drive
    name = os.path.basename(p).lower()
    if name.endswith(".csv") and "gemini" in name:
        merge_upload(p, HELP_CACHE if "help" in name else GEMINI_CACHE)

print(f"\n{len(meta)} images, {meta.photo_id.nunique()} photos, {meta.disease.nunique()} diseases. Results go to {OUT_DIR}")
display(meta.pivot_table(index=["group", "disease_name"], columns="level", values="image", aggfunc="count", fill_value=0))

## 2. Photo tools

The fixes and the 16 quality measurements (the same code as the L3 variants notebook). You don't need to change anything here.

In [ ]:
# ===== SafeCrop photo tools: fixes for noisy leaf photos, and quality measurements =====
# Every function takes and returns a 224 x 224 (or any size) RGB photo as a uint8 numpy array.
import numpy as np
import cv2
from PIL import Image


def to_array(img):
    """PIL image or array -> uint8 RGB array."""
    if isinstance(img, Image.Image):
        img = np.asarray(img.convert("RGB"))
    a = np.asarray(img)
    if a.ndim == 2:
        a = np.stack([a] * 3, axis=2)
    return np.ascontiguousarray(a[..., :3]).astype(np.uint8)


def luminance(a):
    """brightness of every pixel, 0 (black) to 1 (white)"""
    return (a.astype(np.float32) @ np.array([0.299, 0.587, 0.114], np.float32)) / 255.0


def noise_sigma(g):
    """Immerkaer's fast estimate of random noise (same as the loop notebook's), 0 to 1 scale"""
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    conv = cv2.filter2D(g.astype(np.float32), -1, k, borderType=cv2.BORDER_REFLECT)[1:-1, 1:-1]
    H, W = g.shape
    return float(np.sqrt(np.pi / 2) * np.abs(conv).sum() / (6 * (W - 2) * (H - 2)))


# ---------------------------------------------------------------- fixes
def fix_brighten(a, target=0.45, max_gain=4.0):
    """multiply every pixel by the same number so the average brightness reaches `target` (never darkens)"""
    m = float(luminance(a).mean())
    gain = min(max(target / max(m, 1e-3), 1.0), max_gain)
    return np.clip(a.astype(np.float32) * gain, 0, 255).astype(np.uint8)


def fix_gamma(a, target=0.45):
    """gamma curve: lifts dark tones a lot and bright tones a little, so nothing burns out (never darkens)"""
    m = float(np.clip(luminance(a).mean(), 1e-3, 0.999))
    gamma = float(np.clip(np.log(target) / np.log(m), 0.25, 1.0))
    lut = np.clip(255.0 * (np.arange(256) / 255.0) ** gamma, 0, 255).astype(np.uint8)
    return cv2.LUT(a, lut)


def fix_clahe(a, clip=2.0, tiles=8):
    """CLAHE: evens out brightness tile by tile (8 x 8 tiles), with a cap so grain isn't blown up"""
    lab = cv2.cvtColor(a, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def fix_contrast(a, low=1, high=99):
    """stretch the darkest 1% to black and the brightest 1% to white"""
    g = luminance(a) * 255
    lo, hi = np.percentile(g, [low, high])
    if hi - lo < 5:
        return a.copy()
    return np.clip((a.astype(np.float32) - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)


def fix_denoise(a):
    """non-local means: averages each small patch with similar patches elsewhere; strength follows the measured grain"""
    s = noise_sigma(luminance(a)) * 255
    h = float(np.clip(0.9 * s, 3, 30))
    return cv2.fastNlMeansDenoisingColored(a, None, h, h, 7, 21)


def fix_sharpen(a, amount=1.0, sigma=2.0):
    """unsharp mask: adds back the difference between the photo and a blurred copy, so edges look crisper"""
    blur = cv2.GaussianBlur(a, (0, 0), sigma)
    return cv2.addWeighted(a, 1 + amount, blur, -amount, 0)


def fix_smooth(a):
    """edge-preserving smoothing (mean shift): flattens grain into even patches but keeps edges,
    a bit like turning the photo into a drawing (the closest thing to 'vectorising' a photo)"""
    return cv2.pyrMeanShiftFiltering(a, sp=8, sr=24)


def fix_deblock(a):
    """gentle edge-preserving blur that hides JPEG's 8 x 8 blocks"""
    return cv2.bilateralFilter(a, 7, 40, 5)


def fix_zoom(a, keep=0.7):
    """cut out the central 70% and enlarge it back to full size"""
    H, W = a.shape[:2]
    h, w = int(round(H * keep)), int(round(W * keep))
    y, x = (H - h) // 2, (W - w) // 2
    return cv2.resize(a[y:y + h, x:x + w], (W, H), interpolation=cv2.INTER_CUBIC)


def fix_resample(a, factor=0.5):
    """shrink then enlarge: each small square of pixels is averaged, which cancels some grain"""
    H, W = a.shape[:2]
    small = cv2.resize(a, (max(8, int(W * factor)), max(8, int(H * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC)


def fix_white_balance(a):
    """grey world: scale red, green and blue so their averages match, removing a colour cast"""
    f = a.astype(np.float32)
    means = f.reshape(-1, 3).mean(0)
    return np.clip(f * (means.mean() / np.maximum(means, 1e-3)), 0, 255).astype(np.uint8)


def fix_saturation(a, factor=1.5):
    """make colours 50% stronger (low light fades them)"""
    hsv = cv2.cvtColor(a, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 1] = np.clip(hsv[..., 1] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def fix_leaf_crop(a, margin=0.08):
    """zoom to where the green leaf is (instead of a fixed centre cut); falls back to a centre zoom
    when no clear leaf area is found"""
    H, W = a.shape[:2]
    small = cv2.GaussianBlur(a, (0, 0), 3).astype(np.float32) / 255.0
    exg = 2 * small[..., 1] - small[..., 0] - small[..., 2]
    mask = exg > max(0.05, float(np.percentile(exg, 60)))
    if not (0.08 < mask.mean() < 0.9):
        return fix_zoom(a)
    ys, xs = np.nonzero(mask)
    y0, y1 = np.percentile(ys, [2, 98]).astype(int)
    x0, x1 = np.percentile(xs, [2, 98]).astype(int)
    side = int(max(y1 - y0, x1 - x0) * (1 + 2 * margin))
    side = int(np.clip(side, 0.4 * min(H, W), min(H, W)))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    y = int(np.clip(cy - side // 2, 0, H - side))
    x = int(np.clip(cx - side // 2, 0, W - side))
    return cv2.resize(a[y:y + side, x:x + side], (W, H), interpolation=cv2.INTER_CUBIC)


FIXES = {
    "brighten": fix_brighten, "gamma": fix_gamma, "clahe": fix_clahe, "contrast": fix_contrast,
    "denoise": fix_denoise, "sharpen": fix_sharpen, "smooth": fix_smooth, "deblock": fix_deblock,
    "zoom": fix_zoom, "resample": fix_resample, "white_balance": fix_white_balance,
    "saturation": fix_saturation, "leaf_crop": fix_leaf_crop,
}


def apply_chain(a, steps):
    """apply fixes in order, e.g. apply_chain(photo, ["brighten", "zoom"])"""
    a = to_array(a)
    for s in steps:
        a = FIXES[s](a)
    return a


# ---------------------------------------------------------------- quality measurements
FEATURES = ["brightness", "dark_share", "bright_share", "contrast", "dynamic_range", "saturation", "colourfulness",
            "grain", "grain_relative", "colour_grain", "sharpness", "fine_detail", "edges",
            "blur_direction", "blockiness", "leaf_cover"]


def measure(img):
    """16 numbers that describe what is wrong (or right) with a photo"""
    a = to_array(img)
    if max(a.shape[:2]) != 224:
        a = cv2.resize(a, (224, 224), interpolation=cv2.INTER_AREA)
    f = a.astype(np.float32) / 255.0
    g = luminance(a)
    out = {}
    out["brightness"] = float(g.mean())
    out["dark_share"] = float((g < 0.12).mean())
    out["bright_share"] = float((g > 0.95).mean())
    out["contrast"] = float(g.std())
    p2, p98 = np.percentile(g, [2, 98])
    out["dynamic_range"] = float(p98 - p2)
    mx, mn = f.max(2), f.min(2)
    out["saturation"] = float(np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0).mean())
    rg, yb = f[..., 0] - f[..., 1], 0.5 * (f[..., 0] + f[..., 1]) - f[..., 2]
    out["colourfulness"] = float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))
    s = noise_sigma(g)
    out["grain"] = s
    out["grain_relative"] = float(s / max(g.mean(), 0.02))
    ycc = cv2.cvtColor(a, cv2.COLOR_RGB2YCrCb).astype(np.float32) / 255.0
    out["colour_grain"] = float((noise_sigma(ycc[..., 1]) + noise_sigma(ycc[..., 2])) / 2)
    lap = cv2.Laplacian(g, cv2.CV_32F)
    out["sharpness"] = float(np.log(lap.var() + 1e-8))
    P = np.abs(np.fft.fftshift(np.fft.fft2(g - g.mean()))) ** 2
    H, W = g.shape
    yy, xx = np.mgrid[-H // 2:H - H // 2, -W // 2:W - W // 2]
    r = np.hypot(yy / (H / 2), xx / (W / 2))
    band = (r > 0.02) & (r <= 1)
    out["fine_detail"] = float(P[(r > 0.35) & (r <= 1)].sum() / (P[band].sum() + 1e-12))
    u8 = (g * 255).astype(np.uint8)
    med = float(np.median(u8))
    out["edges"] = float((cv2.Canny(u8, int(max(0, 0.66 * med)), int(min(255, 1.33 * med + 1))) > 0).mean())
    # motion blur smears in one direction: compare spectral energy across 12 directions (middle frequencies)
    ang = np.mod(np.arctan2(yy, xx), np.pi)
    mid = (r > 0.08) & (r < 0.5)
    e = np.array([P[mid & (ang >= k * np.pi / 12) & (ang < (k + 1) * np.pi / 12)].mean() for k in range(12)])
    out["blur_direction"] = float(np.log((e.max() + 1e-12) / (e.min() + 1e-12)))
    dh = np.abs(np.diff(g, axis=1))
    dv = np.abs(np.diff(g, axis=0))
    bh = dh[:, 7::8].mean() / (np.delete(dh, np.s_[7::8], axis=1).mean() + 1e-6)
    bv = dv[7::8, :].mean() / (np.delete(dv, np.s_[7::8], axis=0).mean() + 1e-6)
    out["blockiness"] = float((bh + bv) / 2)
    exg = 2 * f[..., 1] - f[..., 0] - f[..., 2]
    out["leaf_cover"] = float((exg > 0.05).mean())
    return out

## 3. Learn what each problem looks like

The 16 measurements are taken on all 840 images, and two random forests learn the problem and the level. They are tested fairly: the photos are split into 5 groups, and each group is checked by forests that never saw any version of those photos. The normal range of each measurement is where 98% of our 210 clean photos fall (the lowest 1% and highest 1% are left out). A wider range would flag too many good photos: with a 90% range, 44% of our clean photos would get at least one warning; with 98%, 16% do, while every L3 photo still does.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
MEAS_FILE = os.path.join(OUT_DIR, "quality_measurements.csv")
F = None
if os.path.exists(MEAS_FILE):
    old = pd.read_csv(MEAS_FILE)
    if list(old.image) == list(meta.image) and set(FEATURES) <= set(old.columns):
        F = old[FEATURES]
        print("Measurements are already saved, so they aren't taken again.")
if F is None:
    F = pd.DataFrame([measure(Image.open(p)) for p in tqdm(meta.path, desc="measuring photos")])
    pd.concat([meta[["image"]], F], axis=1).to_csv(MEAS_FILE, index=False)
meta["problem"] = np.where(meta.level == "L0", "clean", meta.noise)
cv = GroupKFold(n_splits=5)
forest_problem = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1)
forest_level = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1)
oof_problem = cross_val_predict(forest_problem, F[FEATURES], meta.problem, cv=cv, groups=meta.photo_id)
oof_level = cross_val_predict(forest_level, F[FEATURES], meta.level, cv=cv, groups=meta.photo_id)
SHORT = {"clean": "clean", "gaussian_noise": "grain", "low_light": "low light", "jpeg_compression": "compression",
         "motion_blur": "hand-shake blur", "defocus_blur": "out of focus"}
print(f"On photos they never saw: problem right for {(oof_problem == meta.problem).mean() * 100:.1f}% of images, "
      f"level right for {(oof_level == meta.level).mean() * 100:.1f}% "
      f"(within one level: {(np.abs(pd.Series(oof_level).str[1].astype(int) - meta.level.str[1].astype(int)) <= 1).mean() * 100:.1f}%).")
cm = pd.crosstab(meta.problem.map(SHORT), pd.Series(oof_problem).map(SHORT).values, rownames=["real problem"], colnames=["found"])
display(cm)
forest_problem.fit(F[FEATURES], meta.problem)
forest_level.fit(F[FEATURES], meta.level)
importance = pd.Series(forest_problem.feature_importances_, FEATURES).sort_values(ascending=False)
print("Most useful measurements:", ", ".join(f"{k} ({v:.2f})" for k, v in importance.head(6).items()))
RANGES = F[meta.level == "L0"][FEATURES].quantile([0.01, 0.5, 0.99]).T
RANGES.columns = ["low", "typical", "high"]
joblib.dump(dict(problem=forest_problem, level=forest_level, ranges=RANGES, features=FEATURES),
            os.path.join(OUT_DIR, "photo_check_model.joblib"))
print("\nNormal range of each measurement in our clean photos:")
display(RANGES.round(3))

## 4. The fix for each problem

Only one fix is used: **JPEG compression → resample**. In the L3 variants test it took CLIP database from 57% to 79% on compressed photos (9 fixed, 0 broken, p under 0.01).

No other problem gets a fix, because none was proven to help. The best result for each is in the table below. A win of a few photos out of 42 can easily be luck, so the photo is left as it is.

This list is set here on purpose, so the notebook no longer reads `best_fix_per_problem.csv`. That file lists the top scorer for each problem, even when it won by only one photo.

In [ ]:
# Only fixes proven to help CLIP at L3 (exact test, p < 0.05). 42 photos for each problem.
ROUTE = {"clean": [], "low_light": [], "gaussian_noise": [], "jpeg_compression": ["resample"],
         "motion_blur": [], "defocus_blur": []}
EVIDENCE = {
    "clean": "nothing to fix",
    "low_light": "no fix helped: as taken 55%, the best real fix (white balance) 52%",
    "gaussian_noise": "not proven: white balance 26% to 29% (1 fixed, 0 broken, p = 1)",
    "jpeg_compression": "proven: resample 57% to 79% (9 fixed, 0 broken, p under 0.01)",
    "motion_blur": "not proven: resample 33% to 40% (4 fixed, 1 broken, p = 0.38)",
    "defocus_blur": "not proven: white balance 29% to 36% (3 fixed, 0 broken, p = 0.25)",
}
print("Fixes used, from the L3 variants test (CLIP database):")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({"problem": [SHORT[k] for k in ROUTE], "fix": [" + ".join(v) or "no fix" for v in ROUTE.values()],
                          "why": [EVIDENCE[k] for k in ROUTE]}))


## 5. Check photos

For each photo you get: the problems in plain words (with the photo's value and our clean photos' normal range), the main problem and level the forest found (and how sure it is), the chosen fix, and the photo **before and after** with the key measurements again. Everything is saved in `photo_check_report.csv` and one picture per photo in `checked/`.

In [ ]:
SIGNS = [   # measurement, which side is bad, what it means
    ("brightness", "low", "too dark"), ("brightness", "high", "too bright"),
    ("dark_share", "high", "large very dark areas"), ("grain", "high", "grainy (sensor noise)"),
    ("colour_grain", "high", "coloured speckles"), ("sharpness", "low", "blurry"),
    ("fine_detail", "low", "little fine detail"), ("blur_direction", "high", "smeared in one direction (hand shake)"),
    ("blockiness", "high", "blocky (JPEG compression)"), ("saturation", "low", "faded colours"),
    ("dynamic_range", "low", "low contrast (washed out)"), ("leaf_cover", "low", "the leaf fills little of the photo"),
]
KEY_MEASURES = ["brightness", "grain", "sharpness", "fine_detail", "blockiness", "saturation"]
CHECK_DIR = os.path.join(OUT_DIR, "checked")
os.makedirs(CHECK_DIR, exist_ok=True)

def signs_of(m):
    found = []
    for k, side, words in SIGNS:
        lo, hi = RANGES.loc[k, "low"], RANGES.loc[k, "high"]
        if (side == "low" and m[k] < lo) or (side == "high" and m[k] > hi):
            found.append(f"{words}: {m[k]:.3g} (normal {lo:.3g} to {hi:.3g})")
    return found

def route_from_signs(a, m):
    # several problems at once: fix the light first, measure again, then grain, blocks and blur
    low = lambda mm, k: mm[k] < RANGES.loc[k, "low"]
    high = lambda mm, k: mm[k] > RANGES.loc[k, "high"]
    steps = []
    if low(m, "brightness") or high(m, "dark_share"):
        steps += ROUTE["low_light"]
        m = measure(apply_chain(a, steps))
    def add(more):
        steps.extend(s for s in more if s not in steps)
    if high(m, "grain") or high(m, "colour_grain"):
        add(ROUTE["gaussian_noise"])
    if high(m, "blockiness"):
        add(ROUTE["jpeg_compression"])
    if high(m, "blur_direction"):
        add(ROUTE["motion_blur"])
    elif low(m, "sharpness"):
        add(ROUTE["defocus_blur"])
    return steps

def check_photo(path, label=None):
    a = to_array(ImageOps.exif_transpose(Image.open(path)))
    if max(a.shape[:2]) != 224:          # our photos are 224 x 224 central squares: do the same
        h, w = a.shape[:2]
        s = min(h, w)
        a = np.asarray(Image.fromarray(a[(h - s) // 2:(h - s) // 2 + s, (w - s) // 2:(w - s) // 2 + s]).resize((224, 224), Image.BICUBIC))
    m = measure(a)
    x = pd.DataFrame([m])[FEATURES]
    pp = forest_problem.predict_proba(x)[0]
    lp = forest_level.predict_proba(x)[0]
    problem, p_conf = forest_problem.classes_[pp.argmax()], pp.max()
    level, l_conf = forest_level.classes_[lp.argmax()], lp.max()
    if p_conf >= 0.6:
        steps, how = ROUTE.get(problem, []), "chosen for the main problem"
    else:
        steps, how = route_from_signs(a, m), "built from the problems seen, because the forest wasn't sure"
    fixed = apply_chain(a, steps)
    m2 = measure(fixed)
    signs = signs_of(m)
    name = label or os.path.basename(path)
    print(f"\n■ {name}")
    print("  Problems seen:", "; ".join(signs) if signs else "none: every measurement is within our clean photos' range")
    print(f"  Main problem: {SHORT.get(problem, problem)} ({p_conf * 100:.0f}% sure), level {level} ({l_conf * 100:.0f}% sure)")
    if p_conf < 0.6:
        second = forest_problem.classes_[np.argsort(-pp)[1]]
        print(f"  Not sure: it could also be {SHORT.get(second, second)}. Check the list of problems above.")
    if steps:
        print(f"  Fix ({how}):", " + ".join(steps),
              "| after the fix:", ", ".join(f"{k} {m[k]:.3g} → {m2[k]:.3g}" for k in KEY_MEASURES))
    elif problem == "clean" and p_conf >= 0.6:
        print("  Fix: no change needed")
    else:
        print("  Fix: none, because no fix is proven to help with this problem. A clearer photo is the better answer.")
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.9), dpi=130)
    for ax, img, t in [(axes[0], a, "as taken"), (axes[1], fixed, "after: " + (" + ".join(steps) or "no fix used"))]:
        ax.imshow(img); ax.set_title(t, fontsize=9); ax.axis("off")
    fig.suptitle(f"{name}: {SHORT.get(problem, problem)}, level {level}", fontsize=10)
    fig.tight_layout()
    out_png = os.path.join(CHECK_DIR, re.sub(r"[^\w.-]+", "_", os.path.splitext(name)[0]) + ".png")
    fig.savefig(out_png, facecolor="white")
    plt.show() if "inline" in matplotlib.get_backend().lower() else None
    plt.close(fig)
    return dict(photo=name, main_problem=SHORT.get(problem, problem), sure=round(p_conf, 2), level=level,
                level_sure=round(l_conf, 2), problems_seen=" | ".join(signs), fix=" + ".join(steps) or "no fix", fix_chosen=how if steps else "",
                **{f"{k}_before": round(m[k], 4) for k in KEY_MEASURES}, **{f"{k}_after": round(m2[k], 4) for k in KEY_MEASURES})

to_check = []
if SHOW_EXAMPLES:
    for nz in ["low_light", "gaussian_noise", "jpeg_compression", "motion_blur", "defocus_blur"]:
        r = meta[(meta.noise == nz) & (meta.level == "L3")].sample(1, random_state=SEED).iloc[0]
        to_check.append((r.path, f"example {r.image}"))
    r = meta[meta.level == "L0"].sample(1, random_state=SEED).iloc[0]
    to_check.append((r.path, f"example {r.image}"))
if CHECK_FOLDER:
    folder = os.path.join("/content/drive/MyDrive", CHECK_FOLDER)
    if os.path.isdir(folder):
        for f in sorted(os.listdir(folder)):
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp")):
                to_check.append((os.path.join(folder, f), f))
    else:
        print("CHECK_FOLDER not found in your Drive:", folder)
if ASK_FOR_UPLOAD:
    try:
        from google.colab import files
        print("Pick photos to check (or press Cancel to skip):")
        up_dir = os.path.join(WORK, "photo_check_uploads")
        os.makedirs(up_dir, exist_ok=True)
        for name, data in files.upload().items():
            p = os.path.join(up_dir, name)
            with open(p, "wb") as f:
                f.write(data)
            to_check.append((p, name))
    except ImportError:
        print("(No upload button outside Colab; set CHECK_FOLDER instead.)")
    except Exception as e:
        print("Upload skipped:", short(e, 120))

REPORT = []
for path, label in to_check:
    try:
        REPORT.append(check_photo(path, label))
    except Exception as e:
        print(f"\n■ {label}: couldn't be checked ({short(e, 150)})")
if REPORT:
    REPORT = pd.DataFrame(REPORT)
    REPORT.to_csv(os.path.join(OUT_DIR, "photo_check_report.csv"), index=False)
    print(f"\nSaved the report for {len(REPORT)} photos in {OUT_DIR}")
    display(REPORT[["photo", "main_problem", "sure", "level", "fix"]])

## 6. Download

Everything is also in `SafeCrop/results/photo_check` in your Drive.

In [ ]:
zip_path = shutil.make_archive(os.path.join(WORK, "SafeCrop_photo_check"), "zip", OUT_DIR)
print("Zip ready:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download didn't start (" + short(e, 80) + "). Use the folder icon on the left to download it.")